In [ ]:
### Decoing 단계에서 1개의 입력을 처리하기 위해 로드해야 할 Matrix 메모리가 많음
    => memory-bound 인 decodeing(추론) 단계의 메모리 및 지연을 줄이는 목적 (모델 크기는 변화 없음)
    => KV Cache Pruning, Speculative Decoding

### Speculative Decoding
    - 작은 draft 모델이 여러 토큰을 미리 제안
    - target 모델이 한 번의 forward 로 검증
    - chain -> tree -> parallel block 으로 drafting 방식을 확장
    - 출력은 AR decodeing 과 토큰 단위로 동일

    - Vanilla Speculative Decoding
        - 작은 draft 모델 gamma 번 사용해서 예측 (draft)
        - 큰 모델은 1번에 검사 (verify)

    - Tree

    - DFlash

In [ ]:
### Vanilla SD
    - 작은 draft 모델 gamma 번 사용해서 예측 (draft)
    - 큰 모델은 1번에 검사 (verify)

# =====================================================================
# 3-2. Draft Generation
# =====================================================================

@torch.inference_mode()
def draft_tokens(model, x, cache, gamma):
  tokens = []

  # TODO: 다음 draft token을 생성하세요.
  # HINT: greedy_step()을 사용해 draft token을 gamma개 순차적으로 생성하세요.
  for _ in range(gamma):
    x, cache = greedy_step(model, x, cache)
    tokens.append(x)

return torch.cat(tokens, dim=-1), cache


# =====================================================================
# 3-3. Target Verification
# =====================================================================

@torch.inference_mode()
def verify_draft(target, x, drafted, cache):

  # TODO : 마지막 committed token과 draft token들을 하나의 sequence로 연결하세요.
  # HINT: torch.cat()을 사용해 마지막 committed token과 draft token들을 sequence 방향으로 연결하세요.
  verify_ids = torch.cat([x, drafted], dim=-1)

  result = target(
    verify_ids,
    past_key_values=cache,
    use_cache=True
  )

  preds = result.logits[0].argmax(-1)

  return preds, result.past_key_values


# =====================================================================
# 3-4. Accept / Reject
# =====================================================================

def accept_draft(drafted, preds, gamma):

  #TODO: Draft와 Target prediction을 앞에서부터 비교하고 첫 mismatch에서 멈추세요.
  n = 0
  for i in range(gamma):
    if drafted[0, i] == preds[i]:
      n += 1
    else:
      break

  # HINT: accept된 draft token 뒤에 Target의 prediction 하나를 추가하세요.
  new_tokens = torch.cat([drafted[:, :n],preds[n].view(1, 1)],dim=1)

  return new_tokens, n

In [ ]:
### Tree Drafting (EAGLE 계열)
    - Tree 형식의 여러 후보 경로의 draft 를 만든다
    - 이건 그냥 외워야 할듯

In [ ]:
# =====================================================================
# 4-3. [실습 1] Tree Attention Mask
# =====================================================================

import eagle.model.student_tree as student_tree

def build_tree_mask_and_positions(
  mask_index_list,
  total_tokens,
  device=None
):
  # 자기 자신은 항상 볼 수 있음 (대각선 모두 1)
  tree_mask = torch.eye(
    total_tokens + 1,
    dtype=torch.bool,
    device=device
  )
  tree_mask[:, 0] = True    # 0번 (anchor)은 모두가 볼 수 있음 (0번 열 모두 1)

  ######################### 실습 시작 ########################

  # ToDo: child가 parent가 볼 수 있는 node들을 그대로 볼 수 있게 하세요.
  # |= 는 OR 결과를 왼쪽에 저장하는 연산입니다.
  for child, parent in enumerate(mask_index_list, start=1):
    tree_mask[child] |= tree_mask[parent]

  ######################### 실습 끝 ########################

  # 보이는 node 수를 이용해 Tree depth 계산
  tree_position_ids = tree_mask.sum(dim=1).long() - 1
  tree_mask = tree_mask.float()[None, None]
  return tree_mask, tree_position_ids

# 구현을 실제 EAGLE에 연결
student_tree.build_tree_mask_and_positions = build_tree_mask_and_positions

In [ ]:
# =====================================================================
# 4-4. [실습 2] Tree Verification
# =====================================================================

import eagle.model.ea_model as ea_module

def tree_decoding(
  model,
  tree_candidates,
  past_key_values,
  tree_position_ids,
  input_ids,
  retrieve_indices,
):
  position_ids = tree_position_ids + input_ids.shape[1]
  if position_ids.dim() == 1:
    position_ids = position_ids.unsqueeze(0)


  ######################### 실습 시작 ########################

  # To Do: Tree candidate 전체를 Target model에 한 번에 입력하세요.
  outputs, tree_logits, hidden_state = model(
    tree_candidates,
    output_orig=True,
    past_key_values=past_key_values,
    position_ids=position_ids,
  )

  ######################### 실습 끝 ########################

  # 아래 부분은 EAGLE-3 내부 처리를 위해 제공
  if model.use_eagle3:
    device = model.ea_layer.lm_head.weight.device
    states = [x.to(device) for x in outputs["hidden_states"]]
    hidden_state = torch.cat(states, dim=-1)
  logits = tree_logits[0, retrieve_indices]

  return logits, hidden_state, outputs


# 구현을 EAGLE에 연결
ea_module.tree_decoding = tree_decoding

In [ ]:
# =====================================================================
# 4-5. [실습 3] Tree Acceptance
# =====================================================================

def evaluate_posterior(
  logits,
  candidates,
  logits_processor,
):

  # 본 실습에서는 Greedy Decoding만 사용
  assert logits_processor is None

  # 각 branch에서 Draft와 Target prediction 비교
  matches = (
    candidates[:, 1:].to(logits.device)
    == logits[:, :-1].argmax(dim=-1)
  ).int()

  # 첫 mismatch 전까지 연속으로 맞은 token 수
  accept_lengths = torch.cumprod(
    matches,
    dim=1
  ).sum(dim=1)


  ######################### 실습 시작 ########################

  # To Do: 가장 길게 일치한 branch와 그 accept length를 선택하세요.
  best_candidate = accept_lengths.argmax()
  accept_length = accept_lengths[best_candidate]

  ######################### 실습 끝 ########################

  sample_p = logits[
    best_candidate,
    accept_length
  ]

  return best_candidate, accept_length, sample_p


# 구현을 EAGLE에 연결
ea_module.evaluate_posterior = evaluate_posterior

In [ ]:
### DFlash : Block Diffusion Drafing
    - Diffusion 을 사용해서 한번에 drafting

In [ ]:
# =====================================================================
# 5-1. [실습] Parallel Block Drafting
#   한 번의 DFlash forward로 여러 draft token을 동시에 생성합니다.
# =====================================================================

import dflash.student_block as student_block


def parallel_block_draft(
  model,
  target_hidden,
  noise_embedding,
  draft_position_ids,
  past_key_values_draft,
  output_head,
  verify_size,
):

  global LAST_DRAFT

  # DFlash drafter를 한 번만 forward
  draft_hidden = model(
    target_hidden=target_hidden,
    noise_embedding=noise_embedding,
    position_ids=draft_position_ids,
    past_key_values=past_key_values_draft,
    use_cache=True,
  )

  # 실제로 제안할 draft token 수
  num_draft = verify_size - 1


  ######################### 실습 시작 ########################

  # To Do: 마지막 num_draft개의 hidden state만 선택하세요.
  draft_hidden = draft_hidden[:, -num_draft:, :]

  # To Do: model.compute_logits()을 사용해 모든 위치의 vocabulary logits을 계산하세요.
  draft_logits = model.compute_logits(draft_hidden, output_head)

  # To Do: torch.argmax()를 사용해 모든 위치의 token을 동시에 선택하세요.
  draft_tokens = torch.argmax(draft_logits, dim=-1)

  ######################### 실습 끝 ########################

  # 첫 번째 draft block만 저장
  if LAST_DRAFT is None:
    LAST_DRAFT = draft_tokens.detach().clone()

  return draft_tokens

# 구현을 DFlash generation에 연결
student_block.parallel_block_draft = parallel_block_draft